In [1]:
import json
import random
import re
import ast
import random
from collections import defaultdict
from pathlib import Path
from transformers import AutoTokenizer

ABS_REL_DIR = Path("/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/agent_system/environments")
tok_path = ABS_REL_DIR / "tokenizers" / "qwen3"
tok = AutoTokenizer.from_pretrained(tok_path, use_fast=True, local_files_only=True)

/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_jsonl(path: Path):
    data = []
    jsonl_files = sorted(path.glob("*.jsonl"), key=lambda x: int(x.stem) if x.stem.isdigit() else float('inf'))    
    for jsonl_file in jsonl_files:
        with jsonl_file.open("r", encoding="utf-8") as f:
            for line in f:
                if not line:
                    continue
                sample = json.loads(line)
                if sample.get('score', 0) >= 1:
                    data.append(sample)
    
    print(f"Loaded {len(data)} samples from {len(jsonl_files)} files.")
    return data

In [3]:
files = ['alfworld_4', 'alfworld_5', 'alfworld_6', 'alfworld_7']

# (seed, proc_id) -> list of step dicts
traj_groups = defaultdict(list)

for fname in files:
    new_data = load_jsonl(Path("rejection_sampling") / fname)
    for d in new_data:
        # Handle both possible layouts: d['run_info'] or d['info']['run_info']
        run_info = d.get("run_info") or d.get("info", {}).get("run_info")
        if not run_info:
            continue

        seed = run_info.get("seed")
        proc_id = run_info.get("proc_id")
        step = run_info.get("step")
        if seed is None or proc_id is None:
            continue

        traj_groups[(seed, proc_id)].append(d)

# Now pick exactly one trajectory per seed (even if multiple proc_ids per seed)
seed_to_traj = {}  # seed -> list[step dicts]

for (seed, proc_id), traj in traj_groups.items():
    # Sort this trajectory by step so it's in time order
    def _get_step(sample):
        ri = sample.get("run_info") or sample.get("info", {}).get("run_info") or {}
        return ri.get("step", 0)

    traj_sorted = sorted(traj, key=_get_step)

    # Only keep the first trajectory we see for each seed
    if seed not in seed_to_traj:
        seed_to_traj[seed] = traj_sorted

# Flatten back into the same format as load_jsonl: one dict per step
compiled_data = []
for seed, traj in seed_to_traj.items():
    compiled_data.extend(traj)

print(f"Compiled {len(compiled_data)} samples from {len(seed_to_traj)} unique seeds across {len(files)} files.")

Loaded 105 samples from 1 files.
Loaded 351 samples from 2 files.
Loaded 502 samples from 2 files.
Loaded 287 samples from 2 files.
Compiled 906 samples from 80 unique seeds across 4 files.


In [4]:
# # Optionally can print out a sample
# for k,v in data[0].items():
#     print(f"{k}\n{v}\n\n")

In [5]:
# Now converting data to my cleaned sft format
def process_and_write(data, filename, sys_prompt_name):
    cleaned_path = Path("cleaned_sft")
    cleaned_path.mkdir(exist_ok=True)
    out_file = cleaned_path / f"{filename}.jsonl"
    with out_file.open("w", encoding="utf-8") as f:
        for d in data:
            inp = d.get("input", "").strip()
            # Remove the starting 'user\n' and ending '\nassistant'
            inp = inp.removeprefix("user\n").removesuffix("\nassistant").strip()
            out = d.get("output", "").strip()

            chat = [
                ["system", sys_prompt_name],
                ["user", inp],
                ["assistant", out]
            ]

            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

In [6]:
# process_and_write(compiled_data, f"rft_alfworld_{len(compiled_data)}", sys_prompt_name="tw_general.txt")

# Clean our Gold Path Data

In [7]:
def load_jsonl(path: Path):
    data = []
    jsonl_files = sorted(path.glob("*.jsonl"), key=lambda x: int(x.stem) if x.stem.isdigit() else float('inf'))    
    for jsonl_file in jsonl_files:
        with jsonl_file.open("r", encoding="utf-8") as f:
            for line in f:
                sample = json.loads(line)
                data.append(sample)
    
    print(f"Loaded {len(data)} samples from {len(jsonl_files)} files.")
    return data

# load our data in
data = load_jsonl(Path("gold_path/"))

Loaded 30387 samples from 2 files.


### Best Move

In [8]:
bestmove_prompt = """You are an expert at textworld games. You will shortly be provided with the output from an environment step -- your task is to immediately respond with the best action to play. Do not include any tags, just immediately respond with the best action.

Your scenario is:
{env_situation}"""

In [9]:
SAVE_BESTMOVE_DATA = True
best_move_data = []
sys_prompt = "tw_general.txt"
cleaned_path = Path("cleaned_sft")
filename = f"bestmove_{len(data)}.jsonl"
out_path = cleaned_path / filename

if SAVE_BESTMOVE_DATA:
    with out_path.open("w", encoding="utf-8") as f:
        for d in data:
            chat = [
                ["system", sys_prompt],
                ["user", bestmove_prompt.format(env_situation=d["obs"])],
                ["assistant", d["info"]["step_info"]["action"]],
            ]
            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

### Best Line

In [16]:
bestline_prompt = """You are an expert at textworld games. You will shortly be provided with the output from an environment step -- your task is to predict the next {num_steps} steps where one step is an action followed by the environment's observation.

So you should respond with something like the following for 2 steps:
action: take lantern
observation: You pick up the lantern.
action: go to dungeon
observation: You enter the dungeon. It is dark.

Only respond with this information -- no additional tags. You should start with action followed by observation and predict {num_steps} full steps. If your final action leads to a win, you should append ' [WON]' at the end of the final observation.

Your scenario is:
{env_situation}"""

In [17]:
# Start by grouping by trajectories
def group_and_sort(data):
    groups = defaultdict(list)
    for d in data:
        run_info = d["info"]["run_info"]
        key = (run_info["seed"], run_info["proc_id"])
        groups[key].append(d)
    for key in groups:
        groups[key] = sorted(groups[key], key=lambda x: x["info"]["run_info"]["step"])
    group_list = list(groups.values())
    print(f"Total groups: {len(group_list) }")
    return group_list

# Get sorted / grouped by seed + proc_id
sorted_groups = group_and_sort(data)

Total groups: 2200


In [18]:
def group_and_sort(data):
    groups = defaultdict(list)
    for d in data:
        ri = d["info"]["run_info"]
        key = (ri["seed"], ri["proc_id"])
        groups[key].append(d)
    for key in groups:
        groups[key] = sorted(groups[key], key=lambda x: x["info"]["run_info"]["step"])
    group_list = list(groups.values())
    print(f"Total groups: {len(group_list)}")
    return group_list


def build_assistant_response(traj, idxs):
    lines = []
    last_idx = idxs[-1]
    won = traj[last_idx]["info"]["env_provided"]["won"]

    for i in idxs:
        si = traj[i]["info"]["step_info"]
        obs = si["obs"]
        if won and i == last_idx:
            obs += " [WON]"
        lines.append(f"action: {si['action']}")
        lines.append(f"observation: {obs}")

    return "\n".join(lines)


def sample_trajectories(sorted_groups, full_step_sizes=(2, 3), samples_per_group=3):
    min_len, max_len = full_step_sizes
    samples = []

    for traj in sorted_groups:
        if not traj:
            continue

        seg_lengths = [random.randint(min_len, max_len) for _ in range(samples_per_group)]
        candidate_idxs = list(range(len(traj)))
        random.shuffle(candidate_idxs)

        taken = set()
        seg_idx = 0

        for start in candidate_idxs:
            if seg_idx >= samples_per_group:
                break

            length = seg_lengths[seg_idx]
            # env_scenario at `start`, assistant uses steps start+1 .. start+length
            env_idx = start
            first_assist = env_idx + 1
            last_assist = env_idx + length

            if last_assist >= len(traj):
                continue

            used_range = range(env_idx, last_assist + 1)
            if any(i in taken for i in used_range):
                continue

            idxs = list(range(first_assist, last_assist + 1))
            taken.update(used_range)

            env_scenario = traj[env_idx]["obs"] + "\n\nNow predict your next actions and observations."
            assistant_response = build_assistant_response(traj, idxs)

            samples.append(
                {
                    "env_scenario": env_scenario,
                    "num_steps": length,
                    "indices": idxs,
                    "assistant_response": assistant_response,
                }
            )

            seg_idx += 1

    return samples

In [19]:
SAVE_BESTLINE_DATA = True
best_line_data = []
sys_prompt = "tw_general.txt"
cleaned_path = Path("cleaned_sft")
filename = f"bestline_{len(data)}.jsonl"
out_path = cleaned_path / filename

if SAVE_BESTLINE_DATA:
    with out_path.open("w", encoding="utf-8") as f:
        sampled = sample_trajectories(sorted_groups, full_step_sizes=(3, 3), samples_per_group=3)
        for s in sampled:
            chat = [
                ["system", sys_prompt],
                ["user", bestline_prompt.format(num_steps=s['num_steps'], env_situation=s['env_scenario'])],
                ["assistant", s['assistant_response']]
            ]
            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")